In [ ]:
import cv2
import json
import numpy as np

from mtcnn import MTCNN
from tensorflow.keras.models import load_model

from alignment import align_face
from utils import cosine_similarity
from embeddings_database import get_embedding


model = load_model('FaceNetclassification_model.h5')

detector = MTCNN()


with open('face_embeddings_database.json', 'r') as f:
    database_embeddings = json.load(f)



def recognize_person(image_path):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    faces = detector.detect_faces(image_rgb)

    if len(faces) > 0:
        face = faces[0]

        x, y, w, h = face['box']
        keypoints = face['keypoints']

        face_crop = image_rgb[y:y+h, x:x+w]

        aligned_face = align_face(face_crop, keypoints)

        embedding_test = get_embedding(model, aligned_face)

        best_match = None
        highest_similarity = 0.0

        for name, embedding_db in database_embeddings.items():
            embedding_db = np.array(embedding_db)

            similarity = cosine_similarity(
                embedding_test,
                embedding_db
            )

            if similarity > highest_similarity:
                highest_similarity = similarity
                best_match = name

        if highest_similarity < 0.1:
            print('Unknown person')
        else:
            print(
                f'Closest match: {best_match} '
                f'with similarity {highest_similarity:.4f}'
            )

    else:
        print('No face detected')


if __name__ == '__main__':
    recognize_person('test_image.jpg')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
La personne la plus proche est : Aouadhi Moez avec une similarité de 0.1043
